# Diagnóstico de calidad de los datos de Instacart

**Actividad:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación
**Rol:** analista de datos responsable de evaluar la calidad inicial de los datasets del proyecto (desarrollo individual)
**Aprendiz:** Juan Camilo Sarrazola Agudelo

> **Cómo está construido este notebook:** igual que en la EV04, las cifras de las interpretaciones no están escritas a mano.
> Se calculan con código a partir de los DataFrames cargados, de modo que el diagnóstico siempre corresponde a los datos.
> En esta actividad **no se limpia ni se modifica ningún registro**: solo se diagnostica, se interpreta y se documenta.

---
## Momento 1. Reconocimiento del proyecto y de los datasets

### 1.1 Nombre y propósito del proyecto
**Proyecto:** *Explorando Instacart: descubre patrones de compra.*

**Propósito:** describir los hábitos de compra de los clientes de Instacart (plataforma de compras de supermercado) a partir de
sus pedidos: cuándo compran, cada cuánto vuelven a comprar y qué productos repiten, para que el equipo de operaciones
pueda tomar decisiones con información confiable.

### 1.2 Pregunta analítica (recuperada de la EV04)
> **¿Qué hábitos de compra presentan los clientes de Instacart —cuándo compran, cada cuánto vuelven a comprar y qué productos recompran—?**

Se concreta en las tres preguntas de análisis formuladas en la EV04:
1. ¿Cada cuántos días, en promedio, vuelven a comprar los clientes? (`days_since_prior_order`)
2. ¿Qué proporción de los artículos de los pedidos son recompras? (`reordered`)
3. ¿En qué días y horas se concentran los pedidos de los clientes recurrentes? (`order_dow`, `order_hour_of_day`, `order_number`)

### 1.3 Datasets del proyecto y su relación con el problema

| N.º | Archivo | DataFrame | Qué contiene | Relación con la pregunta analítica |
|---|---|---|---|---|
| 1 | `instacart_orders.csv` | `orders` | Un registro por pedido: usuario, número de pedido, día, hora y días desde el pedido anterior. | Responde *cuándo* compran y *cada cuánto* vuelven. |
| 2 | `products.csv` | `products` | Catálogo: identificador, nombre, pasillo y departamento del producto. | Permite saber *qué* productos se compran. |
| 3 | `aisles.csv` | `aisles` | Tabla de consulta `aisle_id` → nombre del pasillo. | Agrupa productos por pasillo. |
| 4 | `departments.csv` | `departments` | Tabla de consulta `department_id` → nombre del departamento. | Agrupa productos por departamento. |
| 5 | `order_products.csv` | `order_products` | Detalle de pedidos: un registro por producto, posición en el carrito y si es recompra. | Responde *qué* se recompra y une pedidos con productos. |

### 1.4 Aspectos de calidad que se deben revisar antes de usar los datos
- **Completitud:** columnas con valores ausentes y si la ausencia tiene una explicación lógica.
- **Tipos de datos:** que cada variable tenga un tipo coherente con lo que representa (enteros, códigos, texto, binarios).
- **Unicidad:** filas completamente duplicadas y claves (`order_id`, `product_id`) repetidas.
- **Validez de rangos:** horas entre 0 y 23, días de la semana entre 0 y 6, `reordered` solo 0 o 1, etc.
- **Consistencia lógica:** que las variables no se contradigan entre sí (por ejemplo, primer pedido sin pedido anterior).
- **Integridad referencial:** que las claves de una tabla existan en la tabla con la que se relacionan.
- **Limitaciones:** información que falta para responder la pregunta analítica.

---
## Momento 2. Preparación del entorno y carga de datos

### 2.1 Librerías y funciones auxiliares

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)


def md(texto):
    # Muestra un texto en formato Markdown dentro del notebook.
    display(Markdown(texto))


def fmt(n):
    # Entero con punto de miles (estilo es-CO): 478967 -> 478.967
    return f"{n:,.0f}".replace(",", ".")


def fmt2(x):
    # Decimal con coma decimal y punto de miles: 11.1017 -> 11,10
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def pct(parte, total):
    # Porcentaje con dos decimales en formato es-CO.
    return fmt2(100 * parte / total)


print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)

Python: 3.13.18
pandas: 3.0.5


### 2.2 Ubicación de los archivos
Los CSV no se versionan en GitHub (ver `README.md`). El notebook los busca primero en `data/raw/` (estructura de referencia de la guía)
y luego en `data/` (estructura usada desde la EV04). Si trabaja en Google Colab, agregue su ruta de Drive a `CANDIDATAS`.

In [2]:
ARCHIVOS = {
    "orders": "instacart_orders.csv",
    "products": "products.csv",
    "aisles": "aisles.csv",
    "departments": "departments.csv",
    "order_products": "order_products.csv",
}

CANDIDATAS = [Path("../data/raw"), Path("../data"), Path("data/raw"), Path("data")]


def encontrar_ruta_datos():
    # Devuelve la primera carpeta que contiene los cinco CSV del proyecto.
    for carpeta in CANDIDATAS:
        if all((carpeta / archivo).exists() for archivo in ARCHIVOS.values()):
            return carpeta
    raise FileNotFoundError(
        "No se encontraron los cinco CSV de Instacart. Cópielos en data/raw/ o data/ (ver README.md)."
    )


RUTA_DATOS = encontrar_ruta_datos()
print("Carpeta de datos:", RUTA_DATOS.as_posix())

Carpeta de datos: ../data/raw


### 2.3 Carga de cada dataset en un DataFrame independiente
Los cinco archivos son texto delimitado que usa **punto y coma (`;`)** como separador, por eso se cargan con `pd.read_csv(..., sep=";")`.

In [3]:
orders = pd.read_csv(RUTA_DATOS / ARCHIVOS["orders"], sep=";")
products = pd.read_csv(RUTA_DATOS / ARCHIVOS["products"], sep=";")
aisles = pd.read_csv(RUTA_DATOS / ARCHIVOS["aisles"], sep=";")
departments = pd.read_csv(RUTA_DATOS / ARCHIVOS["departments"], sep=";")
order_products = pd.read_csv(RUTA_DATOS / ARCHIVOS["order_products"], sep=";")

# Diccionario para recorrer los cinco DataFrames con un solo bucle
dataframes = {
    "orders": orders,
    "products": products,
    "aisles": aisles,
    "departments": departments,
    "order_products": order_products,
}

### 2.4 Verificación de la carga
Se comprueba que cada archivo produjo un DataFrame con más de una columna (si el separador fuera incorrecto, todo quedaría en **una sola columna**)
y con al menos un registro.

In [4]:
verificacion = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "archivo": ARCHIVOS[nombre],
            "tamaño archivo (MB)": round((RUTA_DATOS / ARCHIVOS[nombre]).stat().st_size / 1e6, 2),
            "filas": df.shape[0],
            "columnas": df.shape[1],
            "memoria en pandas (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
            "carga correcta": df.shape[0] > 0 and df.shape[1] > 1,
        }
        for nombre, df in dataframes.items()
    ]
)
assert verificacion["carga correcta"].all(), "Algún archivo no se cargó correctamente: revise el separador."
display(verificacion)
md(f"Los **{len(dataframes)}** datasets se cargaron correctamente y suman **{fmt(verificacion['filas'].sum())}** registros.")

,DataFrame,archivo,tamaño archivo (MB),filas,columnas,memoria en pandas (MB),carga correcta
0,orders,instacart_orders.csv,12.31,478967,6,23.0,True
1,products,products.csv,2.12,49694,4,5.1,True
2,aisles,aisles.csv,0.00,134,2,0.0,True
3,departments,departments.csv,0.00,21,2,0.0,True
4,order_products,order_products.csv,90.03,4545007,4,145.4,True


Los **5** datasets se cargaron correctamente y suman **5.073.823** registros.

---
## Momento 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los ocho comandos de inspección que pide la actividad:

| Comando | Propósito |
|---|---|
| `head()` | Visualizar los primeros registros. |
| `tail()` | Visualizar los últimos registros. |
| `shape` | Identificar el número de filas y columnas. |
| `columns` | Consultar los nombres de las variables. |
| `dtypes` | Reconocer los tipos de datos. |
| `info()` | Examinar la estructura y los valores no nulos. |
| `describe()` | Estadísticas descriptivas de las variables numéricas. |
| `describe(include='object')` | Describir las variables de texto, cuando existan. |

> **Nota sobre pandas 3:** desde pandas 3.0 las columnas de texto se cargan con el tipo **`str`** y ya no como `object`.
> `describe(include='object')` las sigue incluyendo por compatibilidad (pandas emite un aviso de obsolescencia, que aquí se silencia).
> Por eso, en `dtypes` las columnas de texto aparecen como `str`.

In [5]:
def tiene_texto(df):
    # True si el DataFrame tiene al menos una columna de texto (object o str).
    return any(pd.api.types.is_string_dtype(df[col]) or df[col].dtype == object for col in df.columns)


def inspeccionar(nombre, df):
    # Ejecuta los ocho comandos de inspección estructural sobre un DataFrame.
    md(f"### Inspección de `{nombre}`")

    md("**`head()`** · primeros 5 registros")
    display(df.head())

    md("**`tail()`** · últimos 5 registros")
    display(df.tail())

    md("**`shape`** · (filas, columnas)")
    print(df.shape)

    md("**`columns`** · nombres de las variables")
    print(list(df.columns))

    md("**`dtypes`** · tipo de dato de cada columna")
    print(df.dtypes)

    md("**`info()`** · estructura y valores no nulos")
    df.info()

    md("**`describe()`** · variables numéricas")
    display(df.describe())

    md("**`describe(include='object')`** · variables de texto")
    if tiene_texto(df):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            display(df.describe(include="object"))
    else:
        print(f"{nombre} no tiene columnas de texto: este resumen no aplica.")

### 3.1 `orders` (pedidos)

In [6]:
inspeccionar("orders", orders)

### Inspección de `orders`

**`head()`** · primeros 5 registros

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


**`tail()`** · últimos 5 registros

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


**`shape`** · (filas, columnas)

... (7 outputs left out)

**`describe()`** · variables numéricas

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
count,4.789670e+05,478967.000000,478967.000000,478967.000000,478967.000000,450148.000000
mean,1.709947e+06,102966.537475,17.157808,2.775058,13.446676,11.101751
std,9.869480e+05,59525.328991,17.704688,2.045869,4.224986,9.189975
min,4.000000e+00,2.000000,1.000000,0.000000,0.000000,0.000000
25%,8.564685e+05,51414.000000,5.000000,1.000000,10.000000,4.000000
50%,1.709522e+06,102633.000000,11.000000,3.000000,13.000000,7.000000
75%,2.562990e+06,154419.000000,23.000000,5.000000,16.000000,15.000000
max,3.421079e+06,206209.000000,100.000000,6.000000,23.000000,30.000000


**`describe(include='object')`** · variables de texto

orders no tiene columnas de texto: este resumen no aplica.


In [7]:
desc_o = orders.describe()
md(f'''
**Características principales de `orders`:**
- {fmt(orders.shape[0])} pedidos y {orders.shape[1]} columnas, todas numéricas. Los pedidos pertenecen a **{fmt(orders["user_id"].nunique())}** usuarios distintos.
- `order_hour_of_day` va de {desc_o.loc["min", "order_hour_of_day"]:.0f} a {desc_o.loc["max", "order_hour_of_day"]:.0f} y `order_dow` de {desc_o.loc["min", "order_dow"]:.0f} a {desc_o.loc["max", "order_dow"]:.0f}: códigos de hora y de día de la semana.
- `order_number` va de {desc_o.loc["min", "order_number"]:.0f} a {desc_o.loc["max", "order_number"]:.0f} (mediana {desc_o.loc["50%", "order_number"]:.0f}).
- `days_since_prior_order` es la única columna `float64`: tiene `count` = {fmt(desc_o.loc["count", "days_since_prior_order"])} frente a {fmt(len(orders))} filas, y va de {desc_o.loc["min", "days_since_prior_order"]:.0f} a {desc_o.loc["max", "days_since_prior_order"]:.0f} días.
- `orders` no tiene columnas de texto, por eso `describe(include='object')` no aplica.
''')


**Características principales de `orders`:**
- 478.967 pedidos y 6 columnas, todas numéricas. Los pedidos pertenecen a **157.437** usuarios distintos.
- `order_hour_of_day` va de 0 a 23 y `order_dow` de 0 a 6: códigos de hora y de día de la semana.
- `order_number` va de 1 a 100 (mediana 11).
- `days_since_prior_order` es la única columna `float64`: tiene `count` = 450.148 frente a 478.967 filas, y va de 0 a 30 días.
- `orders` no tiene columnas de texto, por eso `describe(include='object')` no aplica.


### 3.2 `products` (catálogo de productos)

In [8]:
inspeccionar("products", products)

### Inspección de `products`

**`head()`** · primeros 5 registros

,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
4,5,Green Chile Anytime Sauce,5,13


**`tail()`** · últimos 5 registros

,product_id,product_name,aisle_id,department_id
49689,49690,HIGH PERFORMANCE ENERGY DRINK,64,7
49690,49691,ORIGINAL PANCAKE & WAFFLE MIX,130,14
49691,49692,ORGANIC INSTANT OATMEAL LIGHT MAPLE BROWN SUGAR,130,14
49692,49693,SPRING WATER BODY WASH,127,11
49693,49694,BURRITO- STEAK & CHEESE,38,1


**`shape`** · (filas, columnas)

... (7 outputs left out)

**`describe()`** · variables numéricas

,product_id,aisle_id,department_id
count,49694.000000,49694.000000,49694.000000
mean,24847.500000,67.773192,11.728478
std,14345.566475,38.317265,5.850314
min,1.000000,1.000000,1.000000
25%,12424.250000,35.000000,7.000000
50%,24847.500000,69.000000,13.000000
75%,37270.750000,100.000000,17.000000
max,49694.000000,134.000000,21.000000


**`describe(include='object')`** · variables de texto

,product_name
count,48436
unique,48436
top,Chocolate Sandwich Cookies
freq,1


In [9]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    desc_p = products.describe(include="object")
md(f'''
**Características principales de `products`:**
- {fmt(products.shape[0])} productos y {products.shape[1]} columnas: tres identificadores enteros y el nombre (`{products["product_name"].dtype}`).
- `product_name` tiene `count` = {fmt(desc_p.loc["count", "product_name"])}: faltan **{fmt(products["product_name"].isna().sum())}** nombres.
- `unique` = {fmt(desc_p.loc["unique", "product_name"])} y `freq` = {desc_p.loc["freq", "product_name"]}: ningún nombre se repite *exactamente*, aunque más adelante se revisa si se repite con otras mayúsculas.
- `aisle_id` va de {products["aisle_id"].min()} a {products["aisle_id"].max()} y `department_id` de {products["department_id"].min()} a {products["department_id"].max()}, coherente con los catálogos de pasillos y departamentos.
''')


**Características principales de `products`:**
- 49.694 productos y 4 columnas: tres identificadores enteros y el nombre (`str`).
- `product_name` tiene `count` = 48.436: faltan **1.258** nombres.
- `unique` = 48.436 y `freq` = 1: ningún nombre se repite *exactamente*, aunque más adelante se revisa si se repite con otras mayúsculas.
- `aisle_id` va de 1 a 134 y `department_id` de 1 a 21, coherente con los catálogos de pasillos y departamentos.


### 3.3 `aisles` (pasillos)

In [10]:
inspeccionar("aisles", aisles)

### Inspección de `aisles`

**`head()`** · primeros 5 registros

,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


**`tail()`** · últimos 5 registros

,aisle_id,aisle
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief
133,134,specialty wines champagnes


**`shape`** · (filas, columnas)

... (7 outputs left out)

**`describe()`** · variables numéricas

,aisle_id
count,134.000000
mean,67.500000
std,38.826537
min,1.000000
25%,34.250000
50%,67.500000
75%,100.750000
max,134.000000


**`describe(include='object')`** · variables de texto

,aisle
count,134
unique,134
top,prepared soups salads
freq,1


In [11]:
md(f'''
**Características principales de `aisles`:**
- Tabla de consulta pequeña: {fmt(aisles.shape[0])} pasillos y {aisles.shape[1]} columnas (`aisle_id` entero y `aisle` texto).
- Los {fmt(aisles["aisle"].nunique())} nombres son únicos y no hay ausentes. Sin embargo, uno de los pasillos se llama **`{aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle"].iloc[0]}`** (id {aisles.loc[aisles["aisle"].str.lower().eq("missing"), "aisle_id"].iloc[0]}): es una categoría de relleno, no un pasillo real.
''')


**Características principales de `aisles`:**
- Tabla de consulta pequeña: 134 pasillos y 2 columnas (`aisle_id` entero y `aisle` texto).
- Los 134 nombres son únicos y no hay ausentes. Sin embargo, uno de los pasillos se llama **`missing`** (id 100): es una categoría de relleno, no un pasillo real.


### 3.4 `departments` (departamentos)

In [12]:
inspeccionar("departments", departments)

### Inspección de `departments`

**`head()`** · primeros 5 registros

,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


**`tail()`** · últimos 5 registros

,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing


**`shape`** · (filas, columnas)

... (7 outputs left out)

**`describe()`** · variables numéricas

,department_id
count,21.000000
mean,11.000000
std,6.204837
min,1.000000
25%,6.000000
50%,11.000000
75%,16.000000
max,21.000000


**`describe(include='object')`** · variables de texto

,department
count,21
unique,21
top,frozen
freq,1


In [13]:
md(f'''
**Características principales de `departments`:**
- {fmt(departments.shape[0])} departamentos y {departments.shape[1]} columnas, sin ausentes ni nombres repetidos.
- Igual que en `aisles`, existe un departamento llamado **`missing`** (id {departments.loc[departments["department"].eq("missing"), "department_id"].iloc[0]}) y otro llamado `other`: categorías genéricas que no describen el producto.
''')


**Características principales de `departments`:**
- 21 departamentos y 2 columnas, sin ausentes ni nombres repetidos.
- Igual que en `aisles`, existe un departamento llamado **`missing`** (id 21) y otro llamado `other`: categorías genéricas que no describen el producto.


### 3.5 `order_products` (detalle de los pedidos)

In [14]:
inspeccionar("order_products", order_products)

### Inspección de `order_products`

**`head()`** · primeros 5 registros

,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


**`tail()`** · últimos 5 registros

,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


**`shape`** · (filas, columnas)

... (7 outputs left out)

**`describe()`** · variables numéricas

,order_id,product_id,add_to_cart_order,reordered
count,4.545007e+06,4.545007e+06,4.544171e+06,4.545007e+06
mean,1.711166e+06,2.558084e+04,8.351669e+00,5.905025e-01
std,9.850955e+05,1.409552e+04,7.080368e+00,4.917411e-01
min,4.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,8.608170e+05,1.353500e+04,3.000000e+00,0.000000e+00
50%,1.711530e+06,2.527200e+04,6.000000e+00,1.000000e+00
75%,2.561311e+06,3.793500e+04,1.100000e+01,1.000000e+00
max,3.421079e+06,4.969400e+04,6.400000e+01,1.000000e+00


**`describe(include='object')`** · variables de texto

order_products no tiene columnas de texto: este resumen no aplica.


In [15]:
desc_op = order_products.describe()
md(f'''
**Características principales de `order_products`:**
- Es el dataset más grande: **{fmt(order_products.shape[0])}** registros y {order_products.shape[1]} columnas numéricas (≈{fmt(order_products.memory_usage(deep=True).sum() / 1e6)} MB en memoria).
- Cada fila es *un producto dentro de un pedido*; abarca {fmt(order_products["order_id"].nunique())} pedidos y {fmt(order_products["product_id"].nunique())} productos distintos.
- `add_to_cart_order` es `float64`, con `count` = {fmt(desc_op.loc["count", "add_to_cart_order"])} (faltan {fmt(order_products["add_to_cart_order"].isna().sum())}) y máximo {desc_op.loc["max", "add_to_cart_order"]:.0f}.
- `reordered` tiene mínimo {desc_op.loc["min", "reordered"]:.0f}, máximo {desc_op.loc["max", "reordered"]:.0f} y media {fmt2(desc_op.loc["mean", "reordered"])}: es una variable binaria; el {pct(order_products["reordered"].sum(), len(order_products))} % de los artículos son recompras.
''')


**Características principales de `order_products`:**
- Es el dataset más grande: **4.545.007** registros y 4 columnas numéricas (≈145 MB en memoria).
- Cada fila es *un producto dentro de un pedido*; abarca 450.046 pedidos y 45.573 productos distintos.
- `add_to_cart_order` es `float64`, con `count` = 4.544.171 (faltan 836) y máximo 64.
- `reordered` tiene mínimo 0, máximo 1 y media 0,59: es una variable binaria; el 59,05 % de los artículos son recompras.


### 3.6 Tabla de diagnóstico estructural
Resumen de la inspección: filas, columnas, tipos de datos y observaciones de cada dataset. La tabla también se guarda en
`reports/diagnostico_estructural.csv` para que el informe técnico use exactamente los mismos resultados.

In [16]:
def resumir_tipos(df):
    # Cuenta cuántas columnas hay de cada tipo, por ejemplo: 'int64 ×5, float64 ×1'.
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo} ×{n}" for tipo, n in conteo.items())


def observar(nombre, df):
    # Observaciones automáticas de la inspección estructural.
    notas = []
    nulos = df.isna().sum()
    for col, n in nulos[nulos > 0].items():
        notas.append(f"{col}: {fmt(n)} ausentes ({pct(n, len(df))} %)")
    for col in df.select_dtypes(include="float").columns:
        if (df[col].dropna() % 1 == 0).all():
            notas.append(f"{col} guarda enteros como float64")
    duplicadas = df.duplicated().sum()
    if duplicadas:
        notas.append(f"{fmt(duplicadas)} filas duplicadas")
    if not notas:
        notas.append("Sin ausentes ni filas duplicadas")
    return "; ".join(notas) + "."


diagnostico_estructural = pd.DataFrame(
    [
        {
            "Dataset": nombre,
            "Archivo": ARCHIVOS[nombre],
            "Filas": df.shape[0],
            "Columnas": df.shape[1],
            "Tipos de datos": resumir_tipos(df),
            "Observaciones": observar(nombre, df),
        }
        for nombre, df in dataframes.items()
    ]
)

RUTA_REPORTES = RUTA_DATOS.parent / "reports" if RUTA_DATOS.name == "data" else RUTA_DATOS.parent.parent / "reports"
RUTA_REPORTES.mkdir(exist_ok=True)
diagnostico_estructural.to_csv(RUTA_REPORTES / "diagnostico_estructural.csv", index=False, encoding="utf-8")

tabla = "| Dataset | Filas | Columnas | Tipos de datos | Observaciones |\n|---|---|---|---|---|\n"
tabla += "\n".join(
    f"| `{r.Dataset}` | {fmt(r.Filas)} | {r.Columnas} | {r[4]} | {r.Observaciones} |"
    for r in diagnostico_estructural.itertuples(index=False)
)
md(tabla)

| Dataset | Filas | Columnas | Tipos de datos | Observaciones |
|---|---|---|---|---|
| `orders` | 478.967 | 6 | int64 ×5, float64 ×1 | days_since_prior_order: 28.819 ausentes (6,02 %); days_since_prior_order guarda enteros como float64; 15 filas duplicadas. |
| `products` | 49.694 | 4 | int64 ×3, str ×1 | product_name: 1.258 ausentes (2,53 %). |
| `aisles` | 134 | 2 | int64 ×1, str ×1 | Sin ausentes ni filas duplicadas. |
| `departments` | 21 | 2 | int64 ×1, str ×1 | Sin ausentes ni filas duplicadas. |
| `order_products` | 4.545.007 | 4 | int64 ×3, float64 ×1 | add_to_cart_order: 836 ausentes (0,02 %); add_to_cart_order guarda enteros como float64. |

---
## Momento 4. Diagnóstico inicial de calidad
Se analizan los resultados de la inspección en cinco frentes: valores faltantes, tipos de datos, inconsistencias,
limitaciones frente a la pregunta analítica y acciones de limpieza propuestas.

### 4.1 Valores faltantes

In [17]:
faltantes = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "columna": col,
            "registros": len(df),
            "ausentes": int(df[col].isna().sum()),
            "% ausentes": round(100 * df[col].isna().mean(), 3),
        }
        for nombre, df in dataframes.items()
        for col in df.columns
    ]
)
md("**Columnas con valores ausentes** (las demás columnas están completas):")
display(faltantes[faltantes["ausentes"] > 0].reset_index(drop=True))

**Columnas con valores ausentes** (las demás columnas están completas):

,DataFrame,columna,registros,ausentes,% ausentes
0,orders,days_since_prior_order,478967,28819,6.017
1,products,product_name,49694,1258,2.531
2,order_products,add_to_cart_order,4545007,836,0.018


Para cada columna con ausentes se busca **por qué** faltan los datos, porque la explicación define la acción de limpieza.

In [18]:
# --- days_since_prior_order: ¿los ausentes son primeros pedidos?
nan_dias = orders[orders["days_since_prior_order"].isna()]
primeros_pedidos = (orders["order_number"] == 1).sum()
nan_son_primeros = (nan_dias["order_number"] == 1).all()

# --- product_name: ¿en qué pasillo y departamento están los productos sin nombre?
sin_nombre = products[products["product_name"].isna()]
pasillos_sin_nombre = aisles[aisles["aisle_id"].isin(sin_nombre["aisle_id"])]["aisle"].tolist()
deptos_sin_nombre = departments[departments["department_id"].isin(sin_nombre["department_id"])]["department"].tolist()
productos_en_missing = (products["aisle_id"] == 100).sum()
articulos_sin_nombre = order_products["product_id"].isin(sin_nombre["product_id"]).sum()

# --- add_to_cart_order: ¿en qué pedidos faltan las posiciones?
tam_pedido = order_products.groupby("order_id").size()
pedidos_nan_carrito = order_products.loc[order_products["add_to_cart_order"].isna(), "order_id"].unique()
tam_con_nan = tam_pedido.loc[pedidos_nan_carrito]
max_posicion = order_products["add_to_cart_order"].max()

md(f'''
**`orders.days_since_prior_order` — {fmt(len(nan_dias))} ausentes ({pct(len(nan_dias), len(orders))} %)**
- {"**Todos**" if nan_son_primeros else "No todos"} los ausentes tienen `order_number` = 1, y hay exactamente {fmt(primeros_pedidos)} primeros pedidos en la tabla.
- **Explicación:** en el primer pedido de un cliente no existe un pedido anterior, así que no hay días que contar. Es un ausente *estructural*, no un error.
- **Cómo afecta:** si se rellenara con 0 se diría que esos clientes compraron dos veces el mismo día, y el promedio de días entre compras bajaría artificialmente.

**`products.product_name` — {fmt(len(sin_nombre))} ausentes ({pct(len(sin_nombre), len(products))} %)**
- Todos los productos sin nombre están en el pasillo `{', '.join(pasillos_sin_nombre)}` y en el departamento `{', '.join(deptos_sin_nombre)}`; el pasillo 100 contiene {fmt(productos_en_missing)} productos, es decir, *solo* productos sin nombre.
- Estos productos aparecen en **{fmt(articulos_sin_nombre)}** registros de `order_products` ({pct(articulos_sin_nombre, len(order_products))} % de los artículos).
- **Cómo afecta:** se pueden contar sus compras y recompras, pero no se puede saber qué producto son ni a qué categoría pertenecen.

**`order_products.add_to_cart_order` — {fmt(order_products["add_to_cart_order"].isna().sum())} ausentes ({pct(order_products["add_to_cart_order"].isna().sum(), len(order_products))} %)**
- Los ausentes están concentrados en solo **{fmt(len(pedidos_nan_carrito))}** pedidos, y todos tienen entre {tam_con_nan.min()} y {tam_con_nan.max()} productos.
- La posición máxima registrada es {max_posicion:.0f}, aunque hay pedidos con hasta {tam_pedido.max()} productos.
- **Explicación probable:** el sistema solo guardó posiciones hasta la {max_posicion:.0f}; a partir de ahí quedaron vacías. Es una **limitación de registro** del origen.
- **Cómo afecta:** el impacto es mínimo para casi cualquier análisis, pero no se puede estudiar el orden de los productos en los carritos muy grandes.
''')


**`orders.days_since_prior_order` — 28.819 ausentes (6,02 %)**
- **Todos** los ausentes tienen `order_number` = 1, y hay exactamente 28.819 primeros pedidos en la tabla.
- **Explicación:** en el primer pedido de un cliente no existe un pedido anterior, así que no hay días que contar. Es un ausente *estructural*, no un error.
- **Cómo afecta:** si se rellenara con 0 se diría que esos clientes compraron dos veces el mismo día, y el promedio de días entre compras bajaría artificialmente.

**`products.product_name` — 1.258 ausentes (2,53 %)**
- Todos los productos sin nombre están en el pasillo `missing` y en el departamento `missing`; el pasillo 100 contiene 1.258 productos, es decir, *solo* productos sin nombre.
- Estos productos aparecen en **9.547** registros de `order_products` (0,21 % de los artículos).
- **Cómo afecta:** se pueden contar sus compras y recompras, pero no se puede saber qué producto son ni a qué categoría pertenecen.

**`order_products.add_to_cart_order` — 836 ausentes (0,02 %)**
- Los ausentes están concentrados en solo **70** pedidos, y todos tienen entre 65 y 127 productos.
- La posición máxima registrada es 64, aunque hay pedidos con hasta 127 productos.
- **Explicación probable:** el sistema solo guardó posiciones hasta la 64; a partir de ahí quedaron vacías. Es una **limitación de registro** del origen.
- **Cómo afecta:** el impacto es mínimo para casi cualquier análisis, pero no se puede estudiar el orden de los productos en los carritos muy grandes.


### 4.2 Tipos de datos
Se compara el tipo que asignó pandas con lo que cada variable representa realmente.

In [19]:
# Qué representa cada columna (definido a partir del Momento 1)
significado = {
    ("orders", "order_id"): "identificador",
    ("orders", "user_id"): "identificador",
    ("orders", "order_number"): "conteo",
    ("orders", "order_dow"): "código de día",
    ("orders", "order_hour_of_day"): "hora",
    ("orders", "days_since_prior_order"): "conteo de días",
    ("products", "product_id"): "identificador",
    ("products", "product_name"): "texto libre",
    ("products", "aisle_id"): "identificador",
    ("products", "department_id"): "identificador",
    ("aisles", "aisle_id"): "identificador",
    ("aisles", "aisle"): "categoría",
    ("departments", "department_id"): "identificador",
    ("departments", "department"): "categoría",
    ("order_products", "order_id"): "identificador",
    ("order_products", "product_id"): "identificador",
    ("order_products", "add_to_cart_order"): "posición",
    ("order_products", "reordered"): "binaria (0/1)",
}


def evaluar_tipo(nombre, col, representa):
    s = dataframes[nombre][col]
    dtype = str(s.dtype)
    if representa in ("identificador", "conteo", "posición", "conteo de días", "hora"):
        if pd.api.types.is_integer_dtype(s):
            return "Sí", "Entero, coherente con lo que representa."
        solo_enteros = (s.dropna() % 1 == 0).all()
        return ("Parcial",
                f"Representa enteros{' y todos sus valores son enteros' if solo_enteros else ''}, pero es {dtype} "
                f"porque tiene {fmt(s.isna().sum())} NaN. Conviene `Int64` (entero que admite nulos).")
    if representa == "código de día":
        return ("Parcial", f"Entero 0–6 sin diccionario de días: es una categoría ordenada, no una cantidad (su promedio no tiene sentido).")
    if representa == "binaria (0/1)":
        valores = sorted(s.unique().tolist())
        return ("Parcial", f"Solo contiene {valores}: se puede convertir a `bool` para dejar explícito que es sí/no.")
    if representa == "categoría":
        return ("Sí", f"Texto ({dtype}) con {s.nunique()} valores únicos; podría usarse `category` para ahorrar memoria.")
    return ("Sí", f"Texto ({dtype}), coherente con un nombre de producto.")


filas_tipos = []
for (nombre, col), representa in significado.items():
    coherente, comentario = evaluar_tipo(nombre, col, representa)
    filas_tipos.append({"DataFrame": nombre, "columna": col, "dtype": str(dataframes[nombre][col].dtype),
                        "representa": representa, "¿coherente?": coherente, "comentario": comentario})
tipos = pd.DataFrame(filas_tipos)
assert len(tipos) == sum(df.shape[1] for df in dataframes.values()), "Falta evaluar alguna columna"
display(tipos)

,DataFrame,columna,dtype,representa,¿coherente?,comentario
0,orders,order_id,int64,identificador,Sí,"Entero, coherente con lo que representa."
1,orders,user_id,int64,identificador,Sí,"Entero, coherente con lo que representa."
2,orders,order_number,int64,conteo,Sí,"Entero, coherente con lo que representa."
3,orders,order_dow,int64,código de día,Parcial,"Entero 0–6 sin diccionario de días: es una categoría ordenada, no una cantidad (su pro..."
4,orders,order_hour_of_day,int64,hora,Sí,"Entero, coherente con lo que representa."
5,orders,days_since_prior_order,float64,conteo de días,Parcial,"Representa enteros y todos sus valores son enteros, pero es float64 porque tiene 28.81..."
6,products,product_id,int64,identificador,Sí,"Entero, coherente con lo que representa."
7,products,product_name,str,texto libre,Sí,"Texto (str), coherente con un nombre de producto."
8,products,aisle_id,int64,identificador,Sí,"Entero, coherente con lo que representa."
9,products,department_id,int64,identificador,Sí,"Entero, coherente con lo que representa."


In [20]:
parciales = tipos[tipos["¿coherente?"] != "Sí"]
md(f'''
**Interpretación de los tipos de datos:** de {len(tipos)} columnas, {fmt((tipos["¿coherente?"] == "Sí").sum())} tienen un tipo coherente y
{len(parciales)} son *parcialmente* coherentes: {", ".join(f"`{r.DataFrame}.{r.columna}`" for r in parciales.itertuples(index=False))}.
- Las columnas `float64` en realidad guardan **enteros**: pandas las convierte a decimal solo porque `NaN` es un valor decimal.
- `order_dow` es un **código**: hacer cálculos como su promedio no tiene interpretación; además, el archivo no dice qué día es el 0.
- Ninguna columna numérica quedó como texto, lo que confirma que el separador `;` se aplicó bien y que no hay símbolos extraños en los números.
''')


**Interpretación de los tipos de datos:** de 18 columnas, 14 tienen un tipo coherente y
4 son *parcialmente* coherentes: `orders.order_dow`, `orders.days_since_prior_order`, `order_products.add_to_cart_order`, `order_products.reordered`.
- Las columnas `float64` en realidad guardan **enteros**: pandas las convierte a decimal solo porque `NaN` es un valor decimal.
- `order_dow` es un **código**: hacer cálculos como su promedio no tiene interpretación; además, el archivo no dice qué día es el 0.
- Ninguna columna numérica quedó como texto, lo que confirma que el separador `;` se aplicó bien y que no hay símbolos extraños en los números.


### 4.3 Posibles inconsistencias

#### a) Filas duplicadas y claves repetidas

In [21]:
claves = {
    "orders": ["order_id"],
    "products": ["product_id"],
    "aisles": ["aisle_id"],
    "departments": ["department_id"],
    "order_products": ["order_id", "product_id"],
}
duplicados = pd.DataFrame(
    [
        {
            "DataFrame": nombre,
            "filas duplicadas completas": int(df.duplicated().sum()),
            "clave": " + ".join(claves[nombre]),
            "claves repetidas": int(df.duplicated(subset=claves[nombre]).sum()),
        }
        for nombre, df in dataframes.items()
    ]
)
display(duplicados)

filas_dup = orders[orders.duplicated(keep=False)].sort_values("order_id")
md(f"**Pedidos duplicados en `orders`** ({fmt(orders.duplicated().sum())} copias sobrantes de {fmt(filas_dup['order_id'].nunique())} pedidos):")
display(filas_dup.head(10))

dias_dup = filas_dup["order_dow"].unique().tolist()
horas_dup = filas_dup["order_hour_of_day"].unique().tolist()
dia_dup = ", ".join(str(v) for v in dias_dup)
hora_dup = ", ".join(str(v) for v in horas_dup)
mismo_momento = orders[(orders["order_dow"].isin(dias_dup)) & (orders["order_hour_of_day"].isin(horas_dup))]
md(f'''
- Los duplicados son filas **idénticas en las {orders.shape[1]} columnas**, no pedidos distintos con el mismo número.
- Todos ocurren en el mismo momento: `order_dow` = {dia_dup} y `order_hour_of_day` = {hora_dup}. En esa franja hay {fmt(len(mismo_momento))} pedidos,
  de los cuales {fmt(mismo_momento.duplicated().sum())} son copias. Esa concentración sugiere un **fallo puntual del sistema** al registrar pedidos, no compras repetidas.
- **Impacto:** cuentan dos veces el mismo pedido e inflan los conteos de esa franja horaria.
''')

,DataFrame,filas duplicadas completas,clave,claves repetidas
0,orders,15,order_id,15
1,products,0,product_id,0
2,aisles,0,aisle_id,0
3,departments,0,department_id,0
4,order_products,0,order_id + product_id,0


**Pedidos duplicados en `orders`** (15 copias sobrantes de 15 pedidos):

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
354993,391768,57671,19,3,2,10.0
371905,391768,57671,19,3,2,10.0
119251,408114,68324,4,3,2,18.0
321100,408114,68324,4,3,2,18.0
394347,467134,63189,21,3,2,2.0
250626,467134,63189,21,3,2,2.0
145574,794638,50898,24,3,2,2.0
99462,794638,50898,24,3,2,2.0
242618,1021560,53767,3,3,2,9.0
311713,1021560,53767,3,3,2,9.0



- Los duplicados son filas **idénticas en las 6 columnas**, no pedidos distintos con el mismo número.
- Todos ocurren en el mismo momento: `order_dow` = 3 y `order_hour_of_day` = 2. En esa franja hay 121 pedidos,
  de los cuales 15 son copias. Esa concentración sugiere un **fallo puntual del sistema** al registrar pedidos, no compras repetidas.
- **Impacto:** cuentan dos veces el mismo pedido e inflan los conteos de esa franja horaria.


#### b) Validez de rangos y coherencia lógica entre variables

In [22]:
reglas = [
    ("orders", "order_hour_of_day entre 0 y 23", ~orders["order_hour_of_day"].between(0, 23)),
    ("orders", "order_dow entre 0 y 6", ~orders["order_dow"].between(0, 6)),
    ("orders", "order_number mayor o igual a 1", orders["order_number"] < 1),
    ("orders", "days_since_prior_order entre 0 y 30 (sin contar NaN)",
     orders["days_since_prior_order"].notna() & ~orders["days_since_prior_order"].between(0, 30)),
    ("orders", "si order_number = 1, days_since_prior_order es NaN",
     (orders["order_number"] == 1) & orders["days_since_prior_order"].notna()),
    ("orders", "si days_since_prior_order es NaN, order_number = 1",
     orders["days_since_prior_order"].isna() & (orders["order_number"] != 1)),
    ("order_products", "reordered solo 0 o 1", ~order_products["reordered"].isin([0, 1])),
    ("order_products", "add_to_cart_order mayor o igual a 1 (sin contar NaN)", order_products["add_to_cart_order"] < 1),
]
validacion = pd.DataFrame(
    [{"DataFrame": df, "regla": regla, "registros que la incumplen": int(mascara.sum())} for df, regla, mascara in reglas]
)

# Regla entre tablas: un artículo no puede ser recompra en el primer pedido del cliente
primer_pedido = order_products.merge(orders.drop_duplicates()[["order_id", "order_number"]], on="order_id", how="inner")
recompra_en_primero = int(((primer_pedido["order_number"] == 1) & (primer_pedido["reordered"] == 1)).sum())
validacion.loc[len(validacion)] = ["order_products + orders", "si order_number = 1, reordered = 0", recompra_en_primero]
display(validacion)

md(f'''
**{"Ninguna regla se incumple" if validacion["registros que la incumplen"].sum() == 0 else "Hay reglas incumplidas"}:** los rangos de horas, días y códigos son válidos
y las variables son coherentes entre sí (ausencia de días ⇔ primer pedido; no hay recompras en el primer pedido).
''')

,DataFrame,regla,registros que la incumplen
0,orders,order_hour_of_day entre 0 y 23,0
1,orders,order_dow entre 0 y 6,0
2,orders,order_number mayor o igual a 1,0
3,orders,days_since_prior_order entre 0 y 30 (sin contar NaN),0
4,orders,"si order_number = 1, days_since_prior_order es NaN",0
5,orders,"si days_since_prior_order es NaN, order_number = 1",0
6,order_products,reordered solo 0 o 1,0
7,order_products,add_to_cart_order mayor o igual a 1 (sin contar NaN),0
8,order_products + orders,"si order_number = 1, reordered = 0",0



**Ninguna regla se incumple:** los rangos de horas, días y códigos son válidos
y las variables son coherentes entre sí (ausencia de días ⇔ primer pedido; no hay recompras en el primer pedido).


#### c) Valores sospechosos en las distribuciones

In [23]:
dias = orders["days_since_prior_order"].value_counts().sort_index()
n_30, n_29 = dias.loc[30.0], dias.loc[29.0]
n_0 = dias.loc[0.0]
max_order_number = orders["order_number"].max()
n_max_order = (orders["order_number"] == max_order_number).sum()

md("**Frecuencia de los últimos valores de `days_since_prior_order`:**")
display(dias.tail(5).to_frame("pedidos"))
md(f'''
- El valor **30** aparece {fmt(n_30)} veces ({pct(n_30, dias.sum())} % de los valores disponibles), frente a {fmt(n_29)} del valor 29.
  Un salto así indica que 30 es un **tope**: probablemente significa «30 días o más», no exactamente 30. Promediar la columna subestima la espera real de esos clientes.
- El valor **0** aparece {fmt(n_0)} veces: clientes con dos pedidos el mismo día. Es posible (un pedido olvidado), pero conviene tenerlo en cuenta.
- `order_number` llega como máximo a {max_order_number} ({fmt(n_max_order)} pedidos con ese valor): podría ser también un límite del historial guardado.
''')

**Frecuencia de los últimos valores de `days_since_prior_order`:**

,pedidos
days_since_prior_order,
26.0,2640
27.0,2986
28.0,3745
29.0,2673
30.0,51338



- El valor **30** aparece 51.338 veces (11,40 % de los valores disponibles), frente a 2.673 del valor 29.
  Un salto así indica que 30 es un **tope**: probablemente significa «30 días o más», no exactamente 30. Promediar la columna subestima la espera real de esos clientes.
- El valor **0** aparece 9.589 veces: clientes con dos pedidos el mismo día. Es posible (un pedido olvidado), pero conviene tenerlo en cuenta.
- `order_number` llega como máximo a 100 (194 pedidos con ese valor): podría ser también un límite del historial guardado.


#### d) Inconsistencias en el texto de los nombres de productos

In [24]:
nombres = products["product_name"].dropna()
nombre_normalizado = nombres.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
repetidos_mayus = nombre_normalizado.duplicated().sum()
todo_mayus = (nombres == nombres.str.upper()).sum()
espacios_extremos = (nombres != nombres.str.strip()).sum()
espacios_dobles = nombres.str.contains("  ", regex=False).sum()

ejemplo = products.loc[nombres[nombre_normalizado.duplicated(keep=False)].index]
ejemplo = ejemplo.assign(_n=nombre_normalizado).sort_values("_n").drop(columns="_n")
md("**Ejemplos de productos cuyo nombre se repite al ignorar mayúsculas y espacios:**")
display(ejemplo.head(6))

md(f'''
- **{fmt(repetidos_mayus)}** nombres se repiten si se ignoran mayúsculas y espacios (por ejemplo, el mismo producto escrito en MAYÚSCULAS y en tipo título),
  con identificadores distintos. Pueden ser el mismo producto registrado dos veces o variantes reales: requiere revisión.
- {fmt(todo_mayus)} nombres están completamente en mayúsculas, {fmt(espacios_dobles)} tienen espacios dobles y {fmt(espacios_extremos)} tienen espacios al inicio o al final.
- **Impacto:** al contar «productos más comprados» por nombre, las ventas de un mismo producto quedarían repartidas en dos filas.
''')

**Ejemplos de productos cuyo nombre se repite al ignorar mayúsculas y espacios:**

,product_id,product_name,aisle_id,department_id
15963,15964,100% Prune Juice,98,7
28059,28060,100% Prune Juice,98,7
31844,31845,18-In-1 Hemp Peppermint Pure-Castile Soap,25,11
23339,23340,18-in-1 Hemp Peppermint Pure-Castile Soap,25,11
37559,37560,4% Milkfat Small Curd Cottage Cheese,108,16
34229,34230,4% Milkfat Small Curd Cottage Cheese,108,16



- **141** nombres se repiten si se ignoran mayúsculas y espacios (por ejemplo, el mismo producto escrito en MAYÚSCULAS y en tipo título),
  con identificadores distintos. Pueden ser el mismo producto registrado dos veces o variantes reales: requiere revisión.
- 54 nombres están completamente en mayúsculas, 403 tienen espacios dobles y 4 tienen espacios al inicio o al final.
- **Impacto:** al contar «productos más comprados» por nombre, las ventas de un mismo producto quedarían repartidas en dos filas.


#### e) Integridad referencial entre tablas

In [25]:
pedidos_sin_detalle = ~orders["order_id"].isin(order_products["order_id"])
integridad = pd.DataFrame(
    [
        ("order_products.order_id existe en orders", int((~order_products["order_id"].isin(orders["order_id"])).sum())),
        ("order_products.product_id existe en products", int((~order_products["product_id"].isin(products["product_id"])).sum())),
        ("products.aisle_id existe en aisles", int((~products["aisle_id"].isin(aisles["aisle_id"])).sum())),
        ("products.department_id existe en departments", int((~products["department_id"].isin(departments["department_id"])).sum())),
        ("orders.order_id tiene detalle en order_products", int(pedidos_sin_detalle.sum())),
    ],
    columns=["relación comprobada", "registros sin correspondencia"],
)
display(integridad)

sin_detalle = orders[pedidos_sin_detalle]
md(f'''
- Todas las claves foráneas apuntan a registros existentes: no hay productos, pasillos ni departamentos «huérfanos».
- Sin embargo, **{fmt(len(sin_detalle))}** pedidos de `orders` ({pct(len(sin_detalle), len(orders))} %) **no tienen productos** en `order_products`.
  Ninguno es un primer pedido (su `order_number` mínimo es {sin_detalle["order_number"].min()}), así que no es un patrón de clientes nuevos:
  `order_products` parece ser una **muestra parcial** del detalle de los pedidos.
- **Impacto:** al unir `orders` con `order_products` esos pedidos desaparecen (unión interna) o quedan sin productos (unión externa); hay que decidirlo de forma explícita.
''')

,relación comprobada,registros sin correspondencia
0,order_products.order_id existe en orders,0
1,order_products.product_id existe en products,0
2,products.aisle_id existe en aisles,0
3,products.department_id existe en departments,0
4,orders.order_id tiene detalle en order_products,28906



- Todas las claves foráneas apuntan a registros existentes: no hay productos, pasillos ni departamentos «huérfanos».
- Sin embargo, **28.906** pedidos de `orders` (6,04 %) **no tienen productos** en `order_products`.
  Ninguno es un primer pedido (su `order_number` mínimo es 4), así que no es un patrón de clientes nuevos:
  `order_products` parece ser una **muestra parcial** del detalle de los pedidos.
- **Impacto:** al unir `orders` con `order_products` esos pedidos desaparecen (unión interna) o quedan sin productos (unión externa); hay que decidirlo de forma explícita.


### 4.4 Limitaciones de los datos frente a la pregunta analítica

In [26]:
md(f'''
| Limitación | Evidencia | Qué parte de la pregunta afecta |
|---|---|---|
| No hay fechas reales, solo día de la semana y hora | Columnas de `orders`: {", ".join(f"`{c}`" for c in orders.columns)} | No se pueden estudiar tendencias por mes, temporadas ni festivos: solo patrones semanales. |
| `order_dow` no tiene diccionario | Valores {sorted(orders["order_dow"].unique().tolist())} sin etiquetas | *Cuándo* compran: no se puede afirmar con certeza qué día es el 0 (domingo o lunes). |
| `days_since_prior_order` tiene tope en 30 | {fmt(n_30)} pedidos con el valor 30 | *Cada cuánto* vuelven: el promedio subestima a los clientes que tardan más de un mes. |
| `order_products` no cubre todos los pedidos | {fmt(len(sin_detalle))} pedidos sin detalle | *Qué recompran*: el análisis de productos se basa en una muestra de pedidos. |
| Productos sin nombre ni categoría | {fmt(len(sin_nombre))} productos en pasillo/departamento `missing` | *Qué recompran*: {fmt(articulos_sin_nombre)} artículos no se pueden identificar. |
| No hay precios, cantidades ni datos de clientes | Ninguna tabla tiene esas columnas | No se puede medir gasto, volumen ni segmentar por tipo de cliente. |
''')


| Limitación | Evidencia | Qué parte de la pregunta afecta |
|---|---|---|
| No hay fechas reales, solo día de la semana y hora | Columnas de `orders`: `order_id`, `user_id`, `order_number`, `order_dow`, `order_hour_of_day`, `days_since_prior_order` | No se pueden estudiar tendencias por mes, temporadas ni festivos: solo patrones semanales. |
| `order_dow` no tiene diccionario | Valores [0, 1, 2, 3, 4, 5, 6] sin etiquetas | *Cuándo* compran: no se puede afirmar con certeza qué día es el 0 (domingo o lunes). |
| `days_since_prior_order` tiene tope en 30 | 51.338 pedidos con el valor 30 | *Cada cuánto* vuelven: el promedio subestima a los clientes que tardan más de un mes. |
| `order_products` no cubre todos los pedidos | 28.906 pedidos sin detalle | *Qué recompran*: el análisis de productos se basa en una muestra de pedidos. |
| Productos sin nombre ni categoría | 1.258 productos en pasillo/departamento `missing` | *Qué recompran*: 9.547 artículos no se pueden identificar. |
| No hay precios, cantidades ni datos de clientes | Ninguna tabla tiene esas columnas | No se puede medir gasto, volumen ni segmentar por tipo de cliente. |


### 4.5 Acciones de limpieza y preparación propuestas
Las acciones **no se ejecutan** en esta actividad; se dejan documentadas para la siguiente fase del proyecto.

1. **Eliminar los pedidos duplicados** de `orders` con `drop_duplicates()` y verificar después que `order_id` sea único.
2. **Mantener como `NaN` los ausentes de `days_since_prior_order`** (son primeros pedidos), convertir la columna a `Int64` y crear una
   variable `es_primer_pedido` para que los promedios excluyan esos casos de forma explícita.
3. **Productos sin nombre:** conservar sus registros para conteos y recompras, pero etiquetarlos como `"Desconocido"` y excluirlos
   de los rankings de productos y del análisis por pasillo o departamento.
4. **`add_to_cart_order`:** convertir a `Int64`, dejar los ausentes como `NaN` y documentar que la posición máxima registrada es 64.
5. **Normalizar `product_name`** (minúsculas, sin espacios dobles ni extremos) en una columna auxiliar y revisar los nombres repetidos
   antes de agrupar por nombre; agrupar por `product_id` mientras tanto.
6. **Convertir tipos:** `reordered` a `bool`, `order_dow` a categoría ordenada (cuando se confirme el diccionario de días),
   `aisle` y `department` a `category`.
7. **Tratar el valor 30 de `days_since_prior_order` como «30 o más»**, preferir la mediana al promedio y reportar la proporción de topes.
8. **Decidir explícitamente el tipo de unión** entre `orders` y `order_products` y documentar que el análisis de productos usa solo
   los pedidos con detalle.

### 4.6 Tabla de hallazgos de calidad
Se construye con las variables calculadas en las secciones anteriores y se guarda en `reports/hallazgos_calidad.csv`.

In [27]:
n_dup = int(orders.duplicated().sum())
hallazgos = pd.DataFrame(
    [
        ("orders", "Valores faltantes en days_since_prior_order",
         f"{fmt(len(nan_dias))} NaN ({pct(len(nan_dias), len(orders))} %); todos con order_number = 1",
         "Si se rellenan con 0 se distorsiona el promedio de días entre compras",
         "Mantener NaN, convertir a Int64 y crear la marca es_primer_pedido"),
        ("orders", "Filas duplicadas",
         f"{fmt(n_dup)} filas idénticas, todas con order_dow = {dia_dup} y hora = {hora_dup}",
         "Conteos inflados en esa franja; order_id deja de ser clave única",
         "drop_duplicates() y verificar unicidad de order_id"),
        ("orders", "Tope en days_since_prior_order",
         f"Valor 30: {fmt(n_30)} pedidos frente a {fmt(n_29)} con 29",
         "El promedio subestima el tiempo entre compras",
         "Tratar 30 como «30 o más» y usar la mediana"),
        ("orders", "Código de día sin diccionario",
         f"order_dow con valores {sorted(orders['order_dow'].unique().tolist())} sin etiquetas",
         "No se puede afirmar qué día de la semana compran más",
         "Confirmar el diccionario con la fuente y convertir a categoría ordenada"),
        ("orders + order_products", "Pedidos sin detalle de productos",
         f"{fmt(len(sin_detalle))} pedidos ({pct(len(sin_detalle), len(orders))} %) no están en order_products",
         "Los análisis de productos usan solo una parte de los pedidos",
         "Elegir y documentar el tipo de unión (inner/left)"),
        ("products", "Valores faltantes en product_name",
         f"{fmt(len(sin_nombre))} NaN, todos en pasillo 100 y departamento 21 ('missing')",
         f"{fmt(articulos_sin_nombre)} artículos comprados no se pueden identificar",
         "Etiquetar como 'Desconocido' y excluir de rankings por nombre y categoría"),
        ("products", "Nombres repetidos con distinto formato",
         f"{fmt(repetidos_mayus)} nombres repetidos al ignorar mayúsculas/espacios; {fmt(espacios_dobles)} con espacios dobles",
         "Ventas de un mismo producto repartidas en varias filas",
         "Normalizar el texto y revisar los repetidos; agrupar por product_id"),
        ("aisles / departments", "Categorías de relleno",
         "Pasillo 'missing' (id 100) y departamento 'missing' (id 21)",
         "Aparecerían como una categoría más en los análisis por grupo",
         "Excluir o etiquetar como 'Sin clasificar' en los reportes"),
        ("order_products", "Valores faltantes en add_to_cart_order",
         f"{fmt(order_products['add_to_cart_order'].isna().sum())} NaN en {fmt(len(pedidos_nan_carrito))} pedidos de más de {max_posicion:.0f} productos",
         "Mínimo; solo impide analizar el orden en carritos muy grandes",
         "Convertir a Int64, conservar NaN y documentar el límite de 64"),
        ("orders / order_products", "Tipos no óptimos",
         "days_since_prior_order y add_to_cart_order en float64; reordered 0/1 en int64",
         "Valores como 30.0 o 1.0; ocupa más memoria y es menos claro",
         "Int64 para enteros con nulos, bool para reordered, category para textos repetidos"),
    ],
    columns=["Dataset", "Problema identificado", "Evidencia encontrada", "Posible impacto", "Acción propuesta"],
)
hallazgos.to_csv(RUTA_REPORTES / "hallazgos_calidad.csv", index=False, encoding="utf-8")

tabla = "| Dataset | Problema identificado | Evidencia encontrada (Pandas) | Posible impacto | Acción propuesta |\n|---|---|---|---|---|\n"
tabla += "\n".join("| " + " | ".join(str(v) for v in fila) + " |" for fila in hallazgos.itertuples(index=False))
md(tabla)

| Dataset | Problema identificado | Evidencia encontrada (Pandas) | Posible impacto | Acción propuesta |
|---|---|---|---|---|
| orders | Valores faltantes en days_since_prior_order | 28.819 NaN (6,02 %); todos con order_number = 1 | Si se rellenan con 0 se distorsiona el promedio de días entre compras | Mantener NaN, convertir a Int64 y crear la marca es_primer_pedido |
| orders | Filas duplicadas | 15 filas idénticas, todas con order_dow = 3 y hora = 2 | Conteos inflados en esa franja; order_id deja de ser clave única | drop_duplicates() y verificar unicidad de order_id |
| orders | Tope en days_since_prior_order | Valor 30: 51.338 pedidos frente a 2.673 con 29 | El promedio subestima el tiempo entre compras | Tratar 30 como «30 o más» y usar la mediana |
| orders | Código de día sin diccionario | order_dow con valores [0, 1, 2, 3, 4, 5, 6] sin etiquetas | No se puede afirmar qué día de la semana compran más | Confirmar el diccionario con la fuente y convertir a categoría ordenada |
| orders + order_products | Pedidos sin detalle de productos | 28.906 pedidos (6,04 %) no están en order_products | Los análisis de productos usan solo una parte de los pedidos | Elegir y documentar el tipo de unión (inner/left) |
| products | Valores faltantes en product_name | 1.258 NaN, todos en pasillo 100 y departamento 21 ('missing') | 9.547 artículos comprados no se pueden identificar | Etiquetar como 'Desconocido' y excluir de rankings por nombre y categoría |
| products | Nombres repetidos con distinto formato | 141 nombres repetidos al ignorar mayúsculas/espacios; 403 con espacios dobles | Ventas de un mismo producto repartidas en varias filas | Normalizar el texto y revisar los repetidos; agrupar por product_id |
| aisles / departments | Categorías de relleno | Pasillo 'missing' (id 100) y departamento 'missing' (id 21) | Aparecerían como una categoría más en los análisis por grupo | Excluir o etiquetar como 'Sin clasificar' en los reportes |
| order_products | Valores faltantes en add_to_cart_order | 836 NaN en 70 pedidos de más de 64 productos | Mínimo; solo impide analizar el orden en carritos muy grandes | Convertir a Int64, conservar NaN y documentar el límite de 64 |
| orders / order_products | Tipos no óptimos | days_since_prior_order y add_to_cart_order en float64; reordered 0/1 en int64 | Valores como 30.0 o 1.0; ocupa más memoria y es menos claro | Int64 para enteros con nulos, bool para reordered, category para textos repetidos |

### 4.7 Conclusión del diagnóstico

In [28]:
total_celdas = sum(df.size for df in dataframes.values())
total_ausentes = sum(int(df.isna().sum().sum()) for df in dataframes.values())
md(f'''
**¿En qué condiciones están los datos para continuar con el proyecto?**

1. **Los datos son utilizables.** Los cinco archivos cargan sin errores, las claves entre tablas son íntegras, los rangos son válidos y
   solo el {pct(total_ausentes, total_celdas)} % de todas las celdas ({fmt(total_ausentes)} de {fmt(total_celdas)}) está vacío.
2. **Los problemas encontrados son pocos y tienen explicación:** los ausentes de días son primeros pedidos, los de nombre son productos
   sin clasificar y los del carrito son un límite del registro. Ninguno obliga a descartar los datos.
3. **Antes de analizar hay que preparar:** eliminar {fmt(n_dup)} duplicados, ajustar tipos, normalizar nombres y decidir cómo tratar los
   productos `missing` y los pedidos sin detalle.
4. **La pregunta analítica se puede responder con matices:** el *cuándo* depende del diccionario de días, el *cada cuánto* debe leerse
   con el tope de 30 días y el *qué recompran* se basa en {fmt(order_products["order_id"].nunique())} pedidos con detalle.
''')


**¿En qué condiciones están los datos para continuar con el proyecto?**

1. **Los datos son utilizables.** Los cinco archivos cargan sin errores, las claves entre tablas son íntegras, los rangos son válidos y
   solo el 0,15 % de todas las celdas (30.913 de 21.252.916) está vacío.
2. **Los problemas encontrados son pocos y tienen explicación:** los ausentes de días son primeros pedidos, los de nombre son productos
   sin clasificar y los del carrito son un límite del registro. Ninguno obliga a descartar los datos.
3. **Antes de analizar hay que preparar:** eliminar 15 duplicados, ajustar tipos, normalizar nombres y decidir cómo tratar los
   productos `missing` y los pedidos sin detalle.
4. **La pregunta analítica se puede responder con matices:** el *cuándo* depende del diccionario de días, el *cada cuánto* debe leerse
   con el tope de 30 días y el *qué recompran* se basa en 450.046 pedidos con detalle.


---
## Verificación final (Momento 6)
Comprobación automática de que el diagnóstico incluye todos los datasets y de que los archivos del informe se generaron.

In [29]:
comprobaciones = {
    "Los 5 datasets están cargados": len(dataframes) == 5,
    "Todos aparecen en el diagnóstico estructural": set(diagnostico_estructural["Dataset"]) == set(dataframes),
    "Todos los tipos de columna fueron evaluados": len(tipos) == sum(df.shape[1] for df in dataframes.values()),
    "Existe reports/diagnostico_estructural.csv": (RUTA_REPORTES / "diagnostico_estructural.csv").exists(),
    "Existe reports/hallazgos_calidad.csv": (RUTA_REPORTES / "hallazgos_calidad.csv").exists(),
    "Ningún DataFrame fue modificado (no hubo limpieza)": all(
        df.shape == (fil, col) for df, (fil, col) in zip(dataframes.values(), verificacion[["filas", "columnas"]].itertuples(index=False))
    ),
}
for descripcion, ok in comprobaciones.items():
    print(("OK   " if ok else "FALLA"), descripcion)
assert all(comprobaciones.values())

OK    Los 5 datasets están cargados
OK    Todos aparecen en el diagnóstico estructural
OK    Todos los tipos de columna fueron evaluados
OK    Existe reports/diagnostico_estructural.csv
OK    Existe reports/hallazgos_calidad.csv
OK    Ningún DataFrame fue modificado (no hubo limpieza)
